# 🎮 Phase 6 - Tasks 6.1 & 6.2: Hybrid Recommender & Explanation Signals Prototyping

> **Mục tiêu của Notebook:**
> 1. **Task 6.1 - Weighted Hybrid Fusion Engine:**
>    - Tích hợp 3 nguồn tri thức: Collaborative Filtering (TruncatedSVD), Content-Based (Semantic Vector Centroid) và Sentiment Analysis (VADER Compound + Positive Ratio).
>    - Chuẩn hóa điểm số đa nguồn (Min-Max Scaling) về cùng miền $[0, 1]$.
>    - Cơ chế trọng số thích ứng (Adaptive Weights) giải quyết triệt để Cold-Start User.
> 2. **Task 6.2 - Explanation Signals & Sentiment Reasons Extraction:**
>    - Trích xuất tín hiệu giải thích đa chiều: Tựa game gốc làm mỏ neo (Anchor Game & Similarity %), Trùng khớp thể loại (Thematic overlap), Đồng thuận cộng đồng (Collaborative consensus) và Đánh giá cộng đồng (Sentiment score).
>    - Trích xuất trích dẫn đánh giá tích cực tiêu biểu (Social Proof Review Quotes) từ văn bản đánh giá thực tế của người chơi.
>    - Tạo cấu trúc dữ liệu giải thích chuẩn hóa (Structured Explanation Schema) sẵn sàng phục vụ cho AI Agent (Phase 7) và UI Streamlit (Phase 8).

In [ ]:
import os
import sys
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sys.path.append("..")
from src.models.collaborative.matrix_factorization import SVDRecommender
from src.models.content_based.recommender import ContentBasedRecommender

# Thiết lập giao diện biểu đồ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120

print("[*] Libraries imported successfully!")

## 1. Tải Dữ liệu Silver/Gold & Các Mô hình Đã Huấn luyện

- **Item Features & Metadata:** 25,612 tựa game.
- **Item Sentiment Profiles:** Thống kê VADER sentiment cho từng game.
- **Pre-trained SVD Model:** TruncatedSVD ($k=64$ latent factors).
- **Content-Based Model:** Sentence-Transformers 384-D item embeddings.
- **Review Sentiment Dataset:** 50,000+ đánh giá có phân tích cảm xúc chi tiết để trích xuất trích dẫn (quote highlights).

In [ ]:
# 1. Tải Content-Based Recommender (kèm Embeddings và Item Features)
cb_model = ContentBasedRecommender(
    embeddings_path="../data/gold/item_embeddings.npy",
    items_path="../data/silver/item_features.parquet"
)

# 2. Tải SVD Collaborative Filtering Model
svd_model = SVDRecommender.load_model("../models/collaborative/svd_recommender.joblib")

# 3. Tải Item Sentiment Profiles & Reviews
df_sentiment = pl.read_parquet("../data/silver/item_sentiment.parquet")
df_reviews = pl.read_parquet("../data/silver/review_sentiment.parquet")

# 4. Tải Interactions để lấy lịch sử người dùng
df_interactions = pl.read_parquet("../data/silver/interactions.parquet")

print(f"[+] Catalog items: {len(cb_model.item_ids):,}")
print(f"[+] SVD Model Users: {len(svd_model.user2idx):,} | Items: {len(svd_model.item2idx):,}")
print(f"[+] Sentiment profiles: {len(df_sentiment):,} items")
print(f"[+] Review sentiment samples: {len(df_reviews):,}")
print(f"[+] Clean interactions: {len(df_interactions):,}")

## 2. Tiền xử lý & Chuẩn hóa Điểm số (Score Normalization)

$$S_{\text{sent}}(i) = 0.6 \cdot \text{positive\_ratio}(i) + 0.4 \cdot \left(\frac{\text{compound}(i) + 1}{2}\right)$$

In [ ]:
# Xây dựng sentiment lookup array đồng bộ theo thứ tự item_ids của catalog
sent_dict = {
    row["parent_asin"]: {
        "pos_ratio": row["positive_review_ratio"],
        "compound": row["avg_sentiment_compound"],
        "reviews": row["review_count"]
    }
    for row in df_sentiment.iter_rows(named=True)
}

item_sentiment_scores = np.zeros(len(cb_model.item_ids), dtype=np.float32)
for i, iid in enumerate(cb_model.item_ids):
    if iid in sent_dict:
        pos = sent_dict[iid]["pos_ratio"]
        comp = sent_dict[iid]["compound"]
        norm_comp = (comp + 1.0) / 2.0  # Scale [-1, 1] -> [0, 1]
        item_sentiment_scores[i] = 0.6 * pos + 0.4 * norm_comp
    else:
        item_sentiment_scores[i] = 0.5  # Neutral default

print(f"[+] Item sentiment score range: min={item_sentiment_scores.min():.4f}, max={item_sentiment_scores.max():.4f}, mean={item_sentiment_scores.mean():.4f}")

## 3. Thuật toán Weighted Hybrid Engine Prototyping (Task 6.1)

Công thức kết hợp có trọng số thích ứng:
$$\text{HybridScore}(u, i) = w_{\text{cf}} \cdot \hat{S}_{\text{cf}}(u, i) + w_{\text{cb}} \cdot \hat{S}_{\text{cb}}(u, i) + w_{\text{sent}} \cdot \hat{S}_{\text{sent}}(i)$$

In [ ]:
def min_max_scale(arr: np.ndarray) -> np.ndarray:
    """Scale array linearly to [0, 1]."""
    min_v = np.min(arr)
    max_v = np.max(arr)
    if max_v > min_v:
        return (arr - min_v) / (max_v - min_v)
    return np.zeros_like(arr)

def get_hybrid_recommendations(
    user_id: str = None,
    liked_item_ids: list = None,
    liked_weights: list = None,
    w_cf: float = 0.50,
    w_cb: float = 0.35,
    w_sent: float = 0.15,
    top_k: int = 10,
    exclude_interacted: bool = True
):
    """
    Compute weighted hybrid recommendations across the catalog.
    """
    n_items = len(cb_model.item_ids)
    interacted_asins = set()
    user_liked_history = []
    
    # 1. Collaborative Filtering Score Vector
    cf_available = False
    scores_cf = np.zeros(n_items, dtype=np.float32)
    if user_id and user_id in svd_model.user2idx:
        u_idx = svd_model.user2idx[user_id]
        u_vec = svd_model.user_factors[u_idx]
        svd_item_scores = np.dot(u_vec, svd_model.item_factors.T)
        for i, iid in enumerate(cb_model.item_ids):
            if iid in svd_model.item2idx:
                scores_cf[i] = svd_item_scores[svd_model.item2idx[iid]]
            else:
                scores_cf[i] = svd_model.global_mean
        scores_cf = min_max_scale(scores_cf)
        cf_available = True
        
        u_hist = df_interactions.filter(pl.col("user_id") == user_id)["parent_asin"].to_list()
        interacted_asins.update(u_hist)
        user_liked_history = u_hist
    
    # 2. Content-Based Score Vector
    scores_cb = np.zeros(n_items, dtype=np.float32)
    if liked_item_ids:
        interacted_asins.update(liked_item_ids)
        user_liked_history = liked_item_ids
        valid_indices = [cb_model.item2idx[iid] for iid in liked_item_ids if iid in cb_model.item2idx]
        if valid_indices:
            w = np.array(liked_weights if liked_weights else [1.0] * len(valid_indices), dtype=np.float32)
            item_vecs = cb_model.embeddings[valid_indices]
            user_centroid = np.sum(item_vecs * w.reshape(-1, 1), axis=0)
            norm = np.linalg.norm(user_centroid)
            if norm > 0:
                user_centroid /= norm
            scores_cb = np.dot(cb_model.embeddings, user_centroid)
            scores_cb = min_max_scale(scores_cb)
    elif user_id and cf_available:
        user_top_items = df_interactions.filter(pl.col("user_id") == user_id).sort("rating", descending=True)["parent_asin"].to_list()[:5]
        valid_indices = [cb_model.item2idx[iid] for iid in user_top_items if iid in cb_model.item2idx]
        if valid_indices:
            user_centroid = np.mean(cb_model.embeddings[valid_indices], axis=0)
            norm = np.linalg.norm(user_centroid)
            if norm > 0:
                user_centroid /= norm
            scores_cb = np.dot(cb_model.embeddings, user_centroid)
            scores_cb = min_max_scale(scores_cb)

    # 3. Sentiment Score Vector
    scores_sent = item_sentiment_scores.copy()

    # 4. Adaptive Weights Fallback for Cold-Start
    if not cf_available:
        effective_w_cf = 0.0
        sum_w = w_cb + w_sent
        effective_w_cb = w_cb / sum_w if sum_w > 0 else 0.70
        effective_w_sent = w_sent / sum_w if sum_w > 0 else 0.30
    else:
        total_w = w_cf + w_cb + w_sent
        effective_w_cf = w_cf / total_w
        effective_w_cb = w_cb / total_w
        effective_w_sent = w_sent / total_w

    # 5. Hybrid Linear Fusion
    hybrid_scores = (
        effective_w_cf * scores_cf +
        effective_w_cb * scores_cb +
        effective_w_sent * scores_sent
    )

    if exclude_interacted and interacted_asins:
        for iid in interacted_asins:
            if iid in cb_model.item2idx:
                hybrid_scores[cb_model.item2idx[iid]] = -np.inf

    top_idx = np.argpartition(hybrid_scores, -top_k)[-top_k:]
    top_idx = top_idx[np.argsort(-hybrid_scores[top_idx])]

    results = []
    for idx in top_idx:
        iid = cb_model.idx2item[idx]
        results.append({
            "parent_asin": iid,
            "title": cb_model.item_titles.get(iid, "Unknown"),
            "category": cb_model.item_categories.get(iid, "Unknown"),
            "hybrid_score": round(float(hybrid_scores[idx]), 4),
            "cf_score": round(float(scores_cf[idx]), 4) if cf_available else 0.0,
            "cb_score": round(float(scores_cb[idx]), 4),
            "sentiment_score": round(float(scores_sent[idx]), 4),
            "avg_rating": cb_model.item_ratings.get(iid, 0.0),
        })
    
    return results, (effective_w_cf, effective_w_cb, effective_w_sent), user_liked_history

## 4. Thử nghiệm Trích xuất Tín hiệu Giải thích (Task 6.2 - Explanation Signals Engine)

Một hệ thống gợi ý AI hiện đại không chỉ đưa ra danh sách đề xuất mà cần giải thích rõ ràng **"Tại sao gợi ý tựa game này?"** (Explainable AI - XAI).

Chúng ta thiết kế 4 chiều giải thích bổ trợ nhau:
1. **Anchor Game Reason (Tựa game mỏ neo tương đồng nhất):**
   $$\text{Anchor} = \arg\max_{j \in \text{Liked}} \cos(e_i, e_j)$$
   Giải thích tựa game nào trong quá khứ của người dùng có phong cách chơi/cốt truyện gần nhất với game được gợi ý.
2. **Thematic Category Overlap (Trùng khớp thể loại):**
   Khớp `main_category` và các nhãn phụ của game với gu sở thích của người dùng.
3. **Collaborative Community Signal (Tín hiệu cộng đồng):**
   Dựa trên điểm dự đoán $S_{\text{cf}}$ và sự tương đồng về hành vi của nhóm người chơi cùng sở thích.
4. **Sentiment & Social Proof Highlight (Trích dẫn đánh giá tích cực thực tế):**
   Tỷ lệ đánh giá tích cực (%) và trích xuất nguyên văn 1 câu nhận xét khen ngợi hay nhất từ cộng đồng người chơi thực tế.

In [ ]:
class RecommendationExplainer:
    """
    Extracts multi-faceted, human-readable explanations and real review highlights for recommendations.
    """
    def __init__(self, cb_model, df_sentiment, df_reviews):
        self.cb = cb_model
        self.sent_dict = {
            row["parent_asin"]: row
            for row in df_sentiment.iter_rows(named=True)
        }
        self.df_reviews = df_reviews
        
        # Index positive reviews by parent_asin for fast quote lookup
        self.cached_quotes = {}
        pos_reviews = df_reviews.filter(pl.col("sentiment_label") == "positive")
        for r in pos_reviews.iter_rows(named=True):
            asin = r["parent_asin"]
            if asin not in self.cached_quotes:
                text = r.get("text", "") or ""
                title = r.get("title", "") or ""
                full_text = f"{title}: {text}" if title else text
                full_text = full_text.replace("<br />", " ").replace("\n", " ").strip()
                if 25 <= len(full_text) <= 180:
                    self.cached_quotes[asin] = full_text

    def find_anchor_game(self, rec_asin: str, user_liked_asins: list) -> tuple:
        """Find the game in user history most similar to the recommended game."""
        if not user_liked_asins or rec_asin not in self.cb.item2idx:
            return None, 0.0
        
        rec_idx = self.cb.item2idx[rec_asin]
        rec_vec = self.cb.embeddings[rec_idx]
        
        best_asin = None
        best_sim = -1.0
        
        for past_asin in user_liked_asins:
            if past_asin in self.cb.item2idx and past_asin != rec_asin:
                past_idx = self.cb.item2idx[past_asin]
                sim = float(np.dot(rec_vec, self.cb.embeddings[past_idx]))
                if sim > best_sim:
                    best_sim = sim
                    best_asin = past_asin
                    
        return best_asin, best_sim

    def get_social_proof_quote(self, rec_asin: str) -> str:
        """Retrieve representative positive review quote for this game."""
        if rec_asin in self.cached_quotes:
            return self.cached_quotes[rec_asin]
        return "Cộng đồng game thủ đánh giá cao lối chơi cuốn hút và đồ họa ấn tượng."

    def explain_recommendation(
        self,
        rec_item: dict,
        user_liked_asins: list,
        is_cold_start: bool = False
    ) -> dict:
        """
        Generate full explanation breakdown for a recommended item.
        """
        rec_asin = rec_item["parent_asin"]
        reasons = []
        
        # 1. Anchor game similarity reason
        anchor_asin, anchor_sim = self.find_anchor_game(rec_asin, user_liked_asins)
        if anchor_asin and anchor_sim > 0.40:
            anchor_title = self.cb.item_titles.get(anchor_asin, "một game bạn từng chơi")
            reasons.append(f"🎯 Tương đồng {anchor_sim*100:.1f}% với game bạn yêu thích: '{anchor_title}'")
        
        # 2. Thematic / Category reason
        cat = rec_item.get("category", "N/A")
        if cat and cat != "Unknown":
            reasons.append(f"🎮 Trùng khớp thể loại yêu thích: {cat}")
            
        # 3. Community / Collaborative reason
        if not is_cold_start and rec_item.get("cf_score", 0) > 0.45:
            reasons.append("👥 Được đông đảo người chơi có cùng gu sở thích với bạn đánh giá rất cao")
            
        # 4. Sentiment reason
        if rec_asin in self.sent_dict:
            s_info = self.sent_dict[rec_asin]
            pos_pct = s_info["positive_review_ratio"] * 100
            reasons.append(f"⭐ {pos_pct:.1f}% đánh giá tích cực trên toàn cộng đồng (Điểm cảm xúc: +{s_info['avg_sentiment_compound']:.2f})")
        else:
            reasons.append(f"⭐ Điểm đánh giá trung bình từ người chơi: {rec_item['avg_rating']}/5.0")
            
        # 5. Highlight review quote
        quote = self.get_social_proof_quote(rec_asin)
        
        return {
            "parent_asin": rec_asin,
            "title": rec_item["title"],
            "hybrid_score": rec_item["hybrid_score"],
            "reasons": reasons,
            "highlight_quote": quote,
        }

explainer = RecommendationExplainer(cb_model, df_sentiment, df_reviews)
print("[+] RecommendationExplainer initialized successfully!")

## 5. Thử nghiệm Trực quan: Giải thích Gợi ý cho Warm User (Case Study 1)

Tạo gợi ý Hybrid và xuất thẻ giải thích chi tiết (Explainable Game Cards).

In [ ]:
# Chọn 1 user có tương tác phong phú
top_users = df_interactions.group_by("user_id").agg(pl.len().alias("count")).sort("count", descending=True)
sample_user_id = top_users["user_id"][5]

recs_warm, eff_w, user_history = get_hybrid_recommendations(
    user_id=sample_user_id,
    w_cf=0.50, w_cb=0.35, w_sent=0.15,
    top_k=4
)

print("=" * 95)
print(f"🎮 THẺ GIẢI THÍCH GỢI Ý CHI TIẾT (WARM USER: {sample_user_id})")
print("=" * 95)

for rank, rec in enumerate(recs_warm, 1):
    explanation = explainer.explain_recommendation(rec, user_liked_asins=user_history, is_cold_start=False)
    print(f"\n🏆 #{rank} [{rec['parent_asin']}] {rec['title']}")
    print(f"   📊 Điểm Hybrid: {rec['hybrid_score']} | CF: {rec['cf_score']} | CB: {rec['cb_score']} | Sent: {rec['sentiment_score']}")
    print("   💡 Lý do gợi ý:")
    for reason in explanation["reasons"]:
        print(f"      • {reason}")
    print(f"   💬 Nhận xét tiêu biểu từ người chơi:")
    print(f"      \"{explanation['highlight_quote']}\"")
    print("-" * 90)

## 6. Thử nghiệm Trực quan: Giải thích Gợi ý cho Cold-Start User (Case Study 2)

Người dùng mới chưa có lịch sử đánh giá, chỉ vừa chọn thích tựa game phiêu lưu nhập vai *Final Fantasy*.

In [ ]:
cold_liked_items = [cb_model.item_ids[10]]  # Final Fantasy item
print(f"🆕 Cold-Start User đã chọn thích: [{cold_liked_items[0]}] {cb_model.item_titles[cold_liked_items[0]]}")

recs_cold, eff_cold_w, cold_history = get_hybrid_recommendations(
    liked_item_ids=cold_liked_items,
    w_cf=0.50, w_cb=0.35, w_sent=0.15,
    top_k=3
)

print("=" * 95)
print("🎮 THẺ GIẢI THÍCH GỢI Ý CHI TIẾT (COLD-START USER)")
print("=" * 95)

for rank, rec in enumerate(recs_cold, 1):
    explanation = explainer.explain_recommendation(rec, user_liked_asins=cold_history, is_cold_start=True)
    print(f"\n🏆 #{rank} [{rec['parent_asin']}] {rec['title']}")
    print(f"   📊 Điểm Hybrid: {rec['hybrid_score']} | CB Sim: {rec['cb_score']} | Sent: {rec['sentiment_score']}")
    print("   💡 Lý do gợi ý:")
    for reason in explanation["reasons"]:
        print(f"      • {reason}")
    print(f"   💬 Nhận xét tiêu biểu từ người chơi:")
    print(f"      \"{explanation['highlight_quote']}\"")
    print("-" * 90)

## 7. Phân bố Điểm số & Tác động của Tín hiệu Giải thích

Khảo sát mối liên hệ giữa điểm tương đồng ngữ nghĩa (Content-Based) và điểm cảm xúc người dùng (Sentiment Score).

In [ ]:
# Trực quan hóa tương quan giữa CB Score và Sentiment Score trong Top 50 gợi ý
recs_top50, _, _ = get_hybrid_recommendations(user_id=sample_user_id, top_k=50)
df_plot = pd.DataFrame(recs_top50)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# Biểu đồ 1: Phân bố điểm số các thành phần
sns.kdeplot(df_plot['cf_score'], ax=ax[0], label='Collaborative Filtering (CF)', fill=True, alpha=0.3)
sns.kdeplot(df_plot['cb_score'], ax=ax[0], label='Content-Based (CB)', fill=True, alpha=0.3)
sns.kdeplot(df_plot['sentiment_score'], ax=ax[0], label='Sentiment Score', fill=True, alpha=0.3)
ax[0].set_title('Phân bố Điểm số các Thành phần (Top-50 Gợi ý)', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Điểm chuẩn hóa [0, 1]')
ax[0].legend(frameon=True)

# Biểu đồ 2: Tương quan giữa CB Score và Điểm Hybrid
scatter = ax[1].scatter(df_plot['cb_score'], df_plot['hybrid_score'], c=df_plot['sentiment_score'], cmap='viridis', s=60, edgecolors='black', linewidth=0.5)
cbar = plt.colorbar(scatter, ax=ax[1])
cbar.set_label('Sentiment Score', fontsize=10)
ax[1].set_title('Tương quan CB Score vs Hybrid Score (Màu sắc: Sentiment)', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Content-Based Similarity Score')
ax[1].set_ylabel('Final Hybrid Score')

plt.tight_layout()
plt.show()

## 8. Tổng kết Đánh giá Tasks 6.1 & 6.2

> ### 💡 Những kết luận quan trọng:
> 1. **Cấu trúc giải thích đa chiều hoàn chỉnh:**
>    - Xác định chính xác tựa game mỏ neo (Anchor Item) người dùng đã từng chơi để tạo sự liên kết tự nhiên.
>    - Trích xuất tỷ lệ đồng thuận tích cực và trích dẫn đánh giá thực tế mang lại sự tin cậy cao (Social Proof).
> 2. **Sẵn sàng cho AI Agent & UI:**
>    - Schema dữ liệu giải thích được chuẩn hóa dưới dạng JSON/Dict, giúp AI Agent có thể dễ dàng lấy thông tin để tạo câu trả lời tự nhiên (Task 7.3) và Streamlit có thể hiển thị trực quan (Task 8.3 & 8.4).
> 
> **Bước kế tiếp (Task 6.3):** Thử nghiệm đa dạng hóa danh mục gợi ý & giảm thiên lệch độ phổ biến (MMR & Intra-List Diversity) trên Notebook.